round 1

In [83]:
import itertools
flax_bids = {30: 30000, 29: 5000, 28: 12000, 27: 28000}
flax_asks = {28: 40000, 31: 20000, 32: 20000, 33: 30000}

mushroom_bids = {20: 43000, 19: 17000, 18: 6000, 17: 5000, 16: 10000, 15: 5000, 14: 10000, 13: 7000}
mushroom_asks = {12: 20000, 13: 25000, 14: 35000, 15: 6000, 16: 5000, 17: 0, 18: 10000, 19: 12000}

def run_book(original_bids, original_asks, my_price, my_qty, is_bid, buyback, fee):
    cleared_volumes = {}

    bids = original_bids.copy()
    asks = original_asks.copy()

    if is_bid:
        if bids.get(my_price, -1) != -1:
            bids[my_price] += my_qty
        else:
            bids[my_price] = my_qty
    else:
        if asks.get(my_price, -1) != -1:
            asks[my_price] += my_qty
        else:
            asks[my_price] = my_qty

    potential_clears = sorted(set(bids.keys()) | set(asks.keys()))

    for clear_price in potential_clears:
        clear_volume = 0
        still_clearing = True
        temp_bids = bids.copy()
        temp_asks = asks.copy()
        while still_clearing:
            if not temp_bids or not temp_asks:
                still_clearing = False
                continue
            best_bid = max(temp_bids)
            best_ask = min(temp_asks)

            if best_bid >= clear_price >= best_ask:
                trade_qty = min(temp_bids[best_bid], temp_asks[best_ask])
                clear_volume += trade_qty
                temp_bids[best_bid] -= trade_qty
                temp_asks[best_ask] -= trade_qty
                if temp_bids[best_bid] == 0:
                    del temp_bids[best_bid]
                if temp_asks[best_ask] == 0:
                    del temp_asks[best_ask]
            else:
                still_clearing = False
        cleared_volumes[clear_price] = clear_volume

    best_clear = max(p for p, v in cleared_volumes.items() if v == max(cleared_volumes.values()))

    if is_bid:
        # All existing bids strictly above my price fill first (fully)
        higher_priority_demand = sum(qty for price, qty in original_bids.items() if price > my_price)

        # Existing bids AT my bid price fill before you (time priority)
        same_level_demand = original_bids.get(my_price, 0)

        # Total supply available at or below clearing price
        available_supply = sum(qty for price, qty in original_asks.items() if price <= best_clear)

        # Supply remaining after higher priority orders
        remaining = max(0, available_supply - higher_priority_demand - same_level_demand)

        # Your fill
        your_fill = min(my_qty, remaining)

        # PnL
        pnl = (buyback - best_clear) * your_fill - fee * your_fill
    else:
        # All existing bids strictly below my price fill first (fully)
        higher_priority_supply = sum(qty for price, qty in original_asks.items() if price < my_price)

        # Existing bids AT my ask price fill before me (time priority)
        same_level_supply = original_asks.get(my_price, 0)

        # Total demand available at or above clearing price
        available_demand = sum(qty for price, qty in original_bids.items() if price >= best_clear)

        # Supply remaining after higher priority orders
        remaining = max(0, available_demand - higher_priority_supply - same_level_supply)

        # Your fill
        your_fill = min(my_qty, remaining)

        # PnL
        pnl = (best_clear - buyback) * your_fill - fee * your_fill

    return best_clear, pnl


In [84]:
import numpy as np
# flax seeds optimisation
my_price = 33
my_qty = 9000
buyback = 30
fee = 0

def optimise_flax():
    best_pnl = 0
    best_params = None
    qty_range = np.arange(5000, 15000, 1)
    max_bid = max(flax_bids)
    min_ask = min(flax_asks)
    bid_range = range(min_ask, max_bid + 1)
    ask_range = range(min_ask - 1, max_bid)
    is_bid = True
    for price, qty in itertools.product(bid_range, qty_range):
        clear, pnl = run_book(flax_bids, flax_asks, price, qty, is_bid, buyback, fee)
        if pnl > best_pnl:
            best_pnl = pnl
            best_params = (price, qty, is_bid)
    is_bid = False
    for price, qty in itertools.product(ask_range, qty_range):
        clear, pnl = run_book(flax_bids, flax_asks, price, qty, is_bid, buyback, fee)
        if pnl > best_pnl:
            best_pnl = pnl
            best_params = (price, qty, is_bid)

    return best_params, best_pnl

print(optimise_flax())

((30, np.int64(9999), True), np.int64(9999))


In [85]:
# mushrooms optimisation
my_price = 20
my_qty = 9000
is_bid = True
buyback = 20
fee = 0.1

def optimise_mushrooms():
    best_pnl = 0
    best_params = None
    qty_range = np.arange(30000, 45000, 1)
    max_bid = max(mushroom_bids)
    min_ask = min(mushroom_asks)
    bid_range = range(min_ask, max_bid + 1)
    ask_range = range(min_ask - 1, max_bid)
    is_bid = True
    for price, qty in itertools.product(bid_range, qty_range):
        clear, pnl = run_book(mushroom_bids, mushroom_asks, price, qty, is_bid, buyback, fee)
        if pnl > best_pnl:
            best_pnl = pnl
            best_params = (price, qty, is_bid)
    is_bid = False
    for price, qty in itertools.product(ask_range, qty_range):
        clear, pnl = run_book(mushroom_bids, mushroom_asks, price, qty, is_bid, buyback, fee)
        if pnl > best_pnl:
            best_pnl = pnl
            best_params = (price, qty, is_bid)

    return best_params, best_pnl

print(optimise_mushrooms())


((19, np.int64(40999), True), np.float64(77898.1))


In [86]:
clear, pnl = run_book(mushroom_bids, mushroom_asks, 20, 58000, True, buyback, fee)
print(clear)
print(pnl)

19
52200.0
